In [ ]:
# Independent Air-NRM large-scale NP/Flow LEAN notebook.
# Generated by build_np_lean_notebook.py from the maintained large-scale CA setup.
# ── large-scale fork ─────────────────────────────────────────────────────────
# A copy of LEAN_LLM_OPT_4.1_Air_NRM.ipynb, edited for the large_scale dataset
# (90 ODs, 268 departures, 4 fare families, per-flight capacity). The small-scale
# notebook at the repository root is untouched; the two are deliberately NOT
# merged for now.
#
# Every path below is written relative to the repository root, so move there
# first. This notebook lives three levels down, hence the three parents.
import os, sys
from pathlib import Path
_ROOT = Path.cwd()
while not (_ROOT / "exp_config.yaml").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
if not (_ROOT / "exp_config.yaml").exists():
    raise RuntimeError("run this notebook from inside the lean-llm-opt-main tree")
os.chdir(_ROOT)
sys.path.insert(0, str(_ROOT))
print("working directory:", os.getcwd())


In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import FAISS
import pandas as pd
from langchain_classic.chains import RetrievalQA
from langchain_classic.agents import initialize_agent, AgentType, Tool
from langchain_core.documents import Document
import openai
from datetime import datetime
import re
from datetime import time as time_class
# ---------------------------------------------------------------------------
# Unified experiment configuration + run logging  (see PATCH_GUIDE.md)
# ---------------------------------------------------------------------------
import os
import leanopt_exp as lx

# API keys come from a .env file at the repository root (see .env.example),
# from the shell environment, or -- if neither is set -- from a hidden prompt.
# Nothing is hard-coded here and nothing is written to the run log.
try:
    from dotenv import load_dotenv
    # override=True: the .env file wins over anything already in the
    # environment. Without it, a stale key left in os.environ by an earlier
    # run silently shadows the value you just edited into .env.
    load_dotenv(override=False)
except ImportError:
    pass

CFG = lx.load_config(
    "exp_config.yaml",
    model_profile='gpt-4.1',
    method='LEAN-LLM-OPT',
    dataset='Air-NRM-NP-LS',
    # Repeats. exp_config sets n_repeats: 3 but nothing was reading it here, so
    # every run landed in the same __r0__ bucket and overwrote the last. Taking
    # it from the environment keeps the default at 0 -- no existing invocation
    # changes -- while
    #     for r in 0 1 2; do LEANOPT_REPEAT=$r python run_notebook.py ...; done
    # produces three separately logged runs that aggregate_runs.py can pool into
    # mean +/- std. Worth doing here: temperature is 0.0 and the seed is fixed,
    # yet instance 13 was wrong in one run and right in the next, so a single
    # run cannot tell a stable failure from a coin flip.
    repeat_index=0,
)
lx.ensure_api_keys(CFG)                          # only for providers this profile uses
user_api_key = os.environ.get("OPENAI_API_KEY", "")   # legacy references

LOG = lx.RunLogger(CFG)
EMBEDDINGS = lx.build_embeddings(CFG)

# Problem Classification

In [ ]:
def Classification_Agent(file_path="Large_Scale_Or_Files/RefData.csv"):
    # Initialize the LLM
    llm1 = lx.build_llm(CFG, "classifier")

    # Load and process the data
    # classification only needs the problem description + its class label
    refdata = lx.load_refdata_docs(CFG)
    refdocuments = refdata

    embeddings = EMBEDDINGS
    vectors1 = FAISS.from_documents(refdocuments, embeddings)
    retriever1 = lx.build_retriever(CFG, vectors1, "refdata")
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm1,
        chain_type="stuff",
        retriever=retriever1,
        return_source_documents=True,
    )

    qa_tool1 = Tool(
        name="FileQA",
        func=qa_chain.invoke,
        description=(
            "Use this tool to answer questions about the problem type of the text. "
            "Provide the question as input, and the tool will retrieve the relevant information from the file and use it to answer the question."
        ),
    )

    few_shot_examples = """

    Question: What is the problem type in operation of the text? Please give the answer directly. Text:There are three best-selling items (P1, P2, P3) on Amazon with the profit w_1,w_2,w_3.There is an independent demand stream for each of the products. The objective of the company is to decide which demands to be fufilled over a ﬁnite sales horizon [0,10] to maximize the total expected revenue from ﬁxed initial inventories. The on-hand inventories for the three items are c_1,c_2,c_3 respectively. During the sales horizon, replenishment is not allowed and there is no any in-transit inventories. Customers who want to purchase P1,P2,P3 arrive at each period accoring to a Poisson process with a_1,a_2,a_3 the arrival rates respectively. Decision variables y_1,y_2,y_3 correspond to the number of requests that the firm plans to fulfill for product 1,2,3. These variables are all positive integers.

    Thought: I need to determine the problem type of the text. I'll use the FileQA tool to retrieve the relevant information.

    Action: FileQA

    Action Input: "What is the problem type in operation of the text? text:There are three best-selling items (P1, P2, P3) on Amazon with the profit w_1, w_2, w_3. ..."

    Observation: The problem type of the text is Network Revenue Management.

    Final Answer: Network Revenue Management.

    """

    prefix = f"""You are a helpful assistant that can answer questions about operation problems.

    Use the following examples as a guide. Always use the FileQA tool when you need to retrieve information from the file:


    {few_shot_examples}

    When you need to find information from the file, use the provided tools. And answer the question by given the answer directly. For example,

    """

    suffix = """

    Begin!

    Question: {input}
    {agent_scratchpad}"""

    agent_pc = initialize_agent(
        tools=[qa_tool1],
        llm=llm1,
        agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
        **lx.agent_kwargs(CFG, prefix.format(few_shot_examples=few_shot_examples), suffix),
    )

    openai.api_request_timeout = 60 
    return agent_pc

def Problemtype(query):
    agent_pc = Classification_Agent(file_path="Large_Scale_Or_Files/RefData.csv")
    category_original=agent_pc.invoke(f"What is the problem type in operation of the text? text:{query}")
    type_output = category_original['output']
    return type_output

# ProblemSolvingModel

## PreProcess: Sharing Functions

In [ ]:
# LARGE-SCALE FORK: reads Test_Dataset/Air_NRM/large_scale/ and additionally
# loads flight_capacity.csv, which the small-scale case does not have (there,
# every flight is 187 seats and the number is hard-coded).
#
# CACHE 1 of 2 -- the CSVs. LoadFiles() is called from New_Vectors_Flight,
# New_Vectors_Demand and retrieve_parameter, i.e. several times per flight per
# query. Re-reading five CSVs each time is pure waste; they never change during
# a run. Same pattern as _SCHEDULE_CACHE in build_air_nrm_inputs.py.
_LS_DIR = 'Test_Dataset/Air_NRM/large_scale'
_FILES_CACHE = None


def LoadFiles():
  """v1, v2, od_demand, flight, flight_capacity -- read once, then reused."""
  global _FILES_CACHE
  if _FILES_CACHE is None:
      _FILES_CACHE = (
          pd.read_csv(f'{_LS_DIR}/v1.csv'),
          pd.read_csv(f'{_LS_DIR}/v2.csv'),
          pd.read_csv(f'{_LS_DIR}/od_demand.csv'),
          pd.read_csv(f'{_LS_DIR}/flight.csv'),
          pd.read_csv(f'{_LS_DIR}/flight_capacity.csv'),
      )
  return _FILES_CACHE


def LoadProducts():
  """Fare families, in the order v1.csv lists them.

  The small-scale notebook hard-codes Eco_flexi and Eco_lite. Here there are
  four, and reading them off the column names means the list can never drift
  from the data file.
  """
  v1 = LoadFiles()[0]
  seen = []
  for column in v1.columns:
      if '*' in column:
          name = column.split('*')[0]
          if name not in seen:
              seen.append(name)
  return seen


In [ ]:
# CACHE 2 of 2 -- the FAISS indexes.
#
# The originals rebuilt the index on every call. Note the `query` argument was
# never used in the body: the index depends only on flight.csv / od_demand.csv,
# so every rebuild produced a byte-identical object at full price. flight.csv
# has 1072 rows here against 38 in the small-scale set, and csv_qa_tool_CA is
# called at least twice per query (few-shot example + the agent's tool call),
# so 15 queries meant 15 x 2 x 1072 = 32,160 embeddings, 31,088 of them
# redundant. Built once, it is 1,072.
#
# This does NOT reduce the number of retrievals: each lookup embeds a different
# query string. That is what the lru_cache below is for.
_FLIGHT_INDEX = None
_DEMAND_INDEX = None


def New_Vectors_Flight(query=None):
  global _FLIGHT_INDEX
  if _FLIGHT_INDEX is None:
      v1, v2, demand, df, capacity = LoadFiles()
      new_docs = []
      for _, row in df.iterrows():
          new_docs.append(Document(
              page_content=f"OD={row['Oneway_OD']},Departure_Time_Flight1={row['Departure Time']},Oneway_Product={row['Oneway_Product']},avg_price={row['Avg Price']}",
              metadata={
                  'OD': row['Oneway_OD'],
                  'time': row['Departure Time'],
                  'product': row['Oneway_Product'],
                  'avg_price': row['Avg Price']
              }
          ))
      _FLIGHT_INDEX = FAISS.from_documents(new_docs, EMBEDDINGS)
  return _FLIGHT_INDEX


def New_Vectors_Demand(query=None):
  global _DEMAND_INDEX
  if _DEMAND_INDEX is None:
      v1, v2, demand, df, capacity = LoadFiles()
      new_docs = []
      for _, row in demand.iterrows():
          new_docs.append(Document(
              page_content=f"OD={row['Oneway_OD']}, avg_pax={row['Avg Pax']}",
              metadata={
                  'OD': row['Oneway_OD'],
                  'avg_pax': row['Avg Pax']
              }
          ))
      _DEMAND_INDEX = FAISS.from_documents(new_docs, EMBEDDINGS)
  return _DEMAND_INDEX


# CACHE 2b -- the retrieval results themselves.
#
# The index cache above removes repeated index builds but not repeated lookups.
# Across the 15 queries there are 898 flight mentions but only 248 distinct
# (OD, departure time) pairs, so the same question gets asked ~3.6x on average.
# Same index + same query string = same document, so memoising is mechanical:
# it changes nothing about what the pipeline does, it just stops paying twice.
# 3,592 retrievals -> 992.
from functools import lru_cache


def _parse_doc_field(content, field):
  """Pull one `field=value` out of a retrieved page_content."""
  for pair in [p.strip().replace('"', "'") for p in re.split(r',\s*(?=\w+=)', content)]:
      key, _, value = pair.partition('=')
      if key == field:
          return value
  return None


# One retrieval per FLIGHT, not per (flight, fare family).
#
# langchain's FAISS filter is applied AFTER the nearest-neighbour search: it
# takes the fetch_k closest documents (default 20), filters those, and returns
# up to k. With the small-scale corpus of 38 documents, 20 is over half of
# everything, so the two rows for a flight were nearly always among them. With
# 1072 documents the 20 nearest to
#     "OD=..., Departure Time=..., Oneway_Product=Eco_standard, avg_price="
# may hold only the wrong fare family -- the first live run returned Eco_lite's
# 239.38 where Eco_standard's 457.46 was asked for -- or none at all, which
# came back as None and died in float().
#
# Two changes. fetch_k is set to the corpus size so the filter is exact rather
# than a filter over an arbitrary 20-document window. And k is the number of
# fare families, so one call returns all of a flight's rows and the product is
# then claimed by name instead of by embedding rank -- the four page_contents
# differ by one word, which is not something cosine distance should be asked to
# adjudicate. Retrieval still narrows; it just no longer decides.
#
# Side effect: 4x fewer retrieval calls (3,592 -> 898 before memoisation).
@lru_cache(maxsize=None)
def retrieve_flight_prices(od, time):
  """{product: avg_price} for one flight, from a single filtered retrieval."""
  products = LoadProducts()
  # k and fetch_k come from exp_config.yaml:retrievers.air_flight_ls, so they
  # reach config_hash() and the run_id. Only the filter is per-call.
  retriever = lx.build_retriever(
      CFG, New_Vectors_Flight(), "air_flight_ls",
      search_kwargs={"filter": {"OD": od, "time": time}})
  docs = retriever.invoke(f"OD={od}, Departure Time={time}, avg_price=")

  found = {}
  for doc in docs:
      name = _parse_doc_field(doc.page_content, 'Oneway_Product')
      value = _parse_doc_field(doc.page_content, 'avg_price')
      if name is not None and value is not None:
          found[name] = value

  missing = [p for p in products if p not in found]
  if missing:
      raise LookupError(
          f"retrieval for {od} {time} returned {sorted(found)} -- no row for "
          f"{missing}. The filter is applied after the nearest-neighbour "
          f"search, so check fetch_k against the corpus size.")
  return found


def retrieve_avg_price(od, time, product):
  """One fare family's avg_price. Claimed by name from the flight's rows."""
  return retrieve_flight_prices(od, time)[product]


# Same treatment as the flight lookup. The demand corpus is 90 documents that
# read "OD=('865', '678'), avg_pax=2279.18" -- near-identical strings differing
# only in six digits. The original searched them with k=1 and no filter at all,
# which asks cosine distance to tell 90 ODs apart. It has metadata, so use it:
# filter exactly on the OD, fetch_k over the whole corpus, and check the answer.
@lru_cache(maxsize=None)
def retrieve_avg_pax(od):
  """avg_pax for one OD, filtered exactly and verified against od_demand.csv."""
  v1, v2, demand, flight_df, capacity = LoadFiles()
  retriever = lx.build_retriever(
      CFG, New_Vectors_Demand(), "air_demand_ls",
      search_kwargs={"filter": {"OD": od}})
  docs = retriever.invoke(f"OD={od}, avg_pax=")
  for doc in docs:
      value = _parse_doc_field(doc.page_content, 'avg_pax')
      if value is None:
          continue
      expected = float(demand.set_index('Oneway_OD')['Avg Pax'][od])
      if abs(float(value) - expected) > 1e-9:
          raise AssertionError(
              f"retrieval returned avg_pax {value} for {od}, but od_demand.csv "
              f"says {expected} -- the search matched a different OD")
      return value
  raise LookupError(f"retrieval found no od_demand row for {od}")


def retrieve_key_information(query):
    flight_pattern = r"\(OD\s*=\s*\(\s*'(\w+)'\s*,\s*'(\w+)'\s*\)\s+AND\s+Departure\s+Time='(\d{1,2}:\d{2})'\)"
    matches = re.findall(flight_pattern, query)

    flight_strings = [f"(OD = ('{origin}', '{destination}') AND Departure Time='{departure_time}')" for origin, destination, departure_time in matches]

    new_query = "Retrieve avg_price, avg_pax, value_list, ratio_list, value_0_list, ratio_0_list, and flight_capacity for the following flights:\n" + ", ".join(flight_strings) + "."

    # Carry the capacity sentences through.
    #
    # The agent calls CName first and feeds its output to CSVQA, so whatever
    # this drops, csv_qa_tool_CA never sees. The original kept only the flight
    # list, which is why the small-scale tool had
    #     else: eco_flex_capacity = 1.2
    # -- a default that silently replaced whatever the question actually said.
    # Here the consumption coefficients differ per query and there are four of
    # them, so they have to survive the rewrite.
    tail = re.search(r"Each flight's capacity is.*", query, re.S)
    if tail:
        new_query += "\n" + tail.group().strip()

    return new_query
def retrieve_time_period(departure_time):
    intervals = {
        '12pm~6pm': (time_class(12, 0), time_class(18, 0)),
        '6pm~10pm': (time_class(18, 0), time_class(22, 0)),
        '10pm~8am': (time_class(22, 0), time_class(8, 0)),
        '8am~12pm': (time_class(8, 0), time_class(12, 0))
    }

    if isinstance(departure_time, str):
        try:
            hours, minutes = map(int, departure_time.split(':'))
            departure_time = time_class(hours, minutes)
        except ValueError:
            raise ValueError("Time format should be: 'HH:MM'")

    for interval_name, (start, end) in intervals.items():
        if start < end:
            if start <= departure_time < end:
                return interval_name
        else:
            if departure_time >= start or departure_time < end:
                return interval_name

    return "Unknown"
def retrieve_parameter(O, time_interval, product):
    """v1 cell, v2 cell, v1 no_purchase, v2 no_purchase for one (OD, window, product).

    The original returned 0 for anything it could not find. With two products
    whose names are far apart that is survivable; with four it is not -- a
    single typo or an unbucketable departure time would silently zero out a
    coefficient and the model would still solve, just wrongly. So every lookup
    that misses now raises.
    """
    v1, v2, demand, df, capacity = LoadFiles()
    if time_interval == "Unknown":
        raise ValueError(f"departure time did not fall in any window for OD={O}")
    key = product + '*' + f'({time_interval})'

    subset = v1[v1['OD Pairs'] == O]
    subset2 = v2[v2['OD Pairs'] == O]
    if subset.empty or subset2.empty:
        raise KeyError(f"OD {O!r} is not a row of v1.csv / v2.csv")
    if key not in subset.columns or key not in subset2.columns:
        raise KeyError(f"column {key!r} is in neither v1.csv nor v2.csv; "
                       f"known products are {LoadProducts()}")

    # float(), not the numpy scalar pandas hands back. These values are printed
    # into the observation string the agent reads, and under numpy 2 a numpy
    # scalar's repr is "np.float64(0.103396)" -- which would land in the prompt
    # verbatim, cost tokens, and give the model something it cannot parse.
    return (float(subset[key].values[0]),
            float(subset2[key].values[0]),
            float(subset['no_purchase'].values[0]),
            float(subset2['no_purchase'].values[0]))


def generate_coefficients(OD, time):
    """All fare families for one (OD, departure time).

    The small-scale version called retrieve_parameter twice, once for Eco_flexi
    and once for Eco_lite, and returned a flat 6-tuple. There are four families
    here, so it returns dicts keyed by product name instead -- the shape no
    longer depends on how many products the dataset happens to have.

        values  {product: v1 cell}      -> the SBLP scale constraint
        ratios  {product: v2 cell}      -> the SBLP balance constraint
        value_0                          v1 no_purchase  (the v_0 the scale
                                         constraint divides by; it is 1 in the
                                         small-scale files and is NOT here)
        ratio_0                          v2 no_purchase
    """
    departure_time = datetime.strptime(time, '%H:%M').time()
    time_interval = retrieve_time_period(departure_time)

    values, ratios = {}, {}
    value_0 = ratio_0 = None
    for product in LoadProducts():
        value, ratio, v0, r0 = retrieve_parameter(OD, time_interval, product)
        values[product] = value
        ratios[product] = ratio
        # Same for every product of an OD; assert rather than quietly overwrite.
        if value_0 is not None and (v0 != value_0 or r0 != ratio_0):
            raise AssertionError(f"no_purchase differs across products for {OD}")
        value_0, ratio_0 = v0, r0

    return values, ratios, value_0, ratio_0


def clean_text_preserve_newlines(text):
    cleaned = re.sub(r'\x1b\[[0-9;]*[mK]', '', text)
    cleaned = re.sub(r'[^\x20-\x7E\n]', '', cleaned)
    cleaned = re.sub(r'(\n\s+)(\w+\s*=)', r'\n\2', cleaned)
    cleaned = re.sub(r'\[\s+', '[', cleaned)
    cleaned = re.sub(r'\s+\]', ']', cleaned)
    cleaned = re.sub(r',\s+', ', ', cleaned)

    return cleaned


## Flow

In [ ]:
# LARGE-SCALE NP/FLOW: deterministic five-file data assembly plus LEAN tools.
from Test_Dataset.Air_NRM.large_scale.np_lean_workflow import install_np_workflow

_NP_WORKFLOW = install_np_workflow(globals())
globals().update(_NP_WORKFLOW)


## CA

In [ ]:
# CA-only modeling workflow intentionally omitted from this NP-only notebook.


# Combined Function

In [ ]:
def _is_airnrm_category(category):
    # Local models vary capitalization, hyphen style and the SBLP class name.
    normalized = re.sub(r"[\u2010-\u2015\u2212]", "-", str(category)).casefold()
    return bool(re.search(
        r"\bsales[-\s]+based\s+linear\s+programming\b|\bsblp\b|"
        r"\b(?:network\s+)?revenue\s+management\b", normalized
    ))


def Process_Input(query):
    with lx.stage("classification"):
        category_original = Problemtype(query)
    print(f"Problem type classification finished, it belongs to {category_original}.")
    if not _is_airnrm_category(category_original):
        raise ValueError(f"unroutable problem type: {category_original!r}")
    if "flow conservation constraint" not in query.lower():
        raise ValueError("this NP-only notebook requires a flow-conservation query")
    print("----------Large-scale NP / Flow Constraints----------")
    with lx.stage("modeling"):
        output_model = ProcessPolicyFlow(query)
    return "Policy_Flow", output_model


# Run the test

In [ ]:
import ast as _ast
import re as _re
from tqdm import tqdm


_FENCE_RE = _re.compile(
    r"```([A-Za-z0-9_+-]*)\s*\n(.*?)```", _re.I | _re.S
)
_PARAMETER_NAMES = (
    "products", "flight_info", "avg_price", "value_list", "ratio_list",
    "value_0_list", "ratio_0_list", "avg_pax", "capacity_consum",
    "flight_capacity", "option_num",
)


def _extract_gurobipy_code(text, query=None):
    """Extract and validate LLM code verbatim; never repair or inject data."""
    if not text:
        raise ValueError("LLM returned no answer")
    objective_heading = str(text).find("### Objective Function")
    constraints_heading = str(text).find("### Constraints")
    code_heading = str(text).find("### Generated Code")
    fence_start = str(text).find("```", code_heading if code_heading >= 0 else 0)
    if not 0 <= objective_heading < constraints_heading < code_heading < fence_start:
        raise ValueError(
            "LLM answer must contain Objective Function, Constraints, then Generated Code"
        )
    if "### Retrieved Information" in str(text):
        raise ValueError("LLM answer duplicated parameters outside Generated Code")
    blocks = [
        (match.group(1).lower(), match.group(2).strip(), match.start())
        for match in _FENCE_RE.finditer(str(text))
    ]
    python_blocks = [block for block in blocks if block[0] in ("python", "py")]
    if len(python_blocks) != 1:
        raise ValueError(
            f"expected exactly one tagged Python code block, got {len(python_blocks)}"
        )
    _, code, python_fence_start = python_blocks[0]
    if python_fence_start <= code_heading:
        raise ValueError("tagged Python block is not under ### Generated Code")
    if "gurobipy" not in code or "Model(" not in code:
        raise ValueError("LLM code contains no Gurobi model")
    if "..." in code:
        raise ValueError("LLM code contains an ellipsis placeholder")
    tree = _ast.parse(code)
    if any(isinstance(node, _ast.Constant) and node.value is Ellipsis
           for node in _ast.walk(tree)):
        raise ValueError("LLM code contains an ellipsis placeholder")

    assignments = {}
    for node in tree.body:
        if (isinstance(node, _ast.Assign)
                and len(node.targets) == 1
                and isinstance(node.targets[0], _ast.Name)
                and node.targets[0].id in _PARAMETER_NAMES):
            try:
                assignments[node.targets[0].id] = _ast.literal_eval(node.value)
            except (ValueError, TypeError) as exc:
                raise ValueError(
                    f"parameter {node.targets[0].id} is not a complete literal"
                ) from exc
    missing = [name for name in _PARAMETER_NAMES if name not in assignments]
    if missing:
        raise ValueError(f"LLM code is missing parameters: {missing}")

    # Verification only: compare with source data without changing the LLM code.
    if query is not None:
        expected = NP_DATA.assemble(query)
        different = [name for name in _PARAMETER_NAMES
                     if assignments[name] != expected[name]]
        if different:
            raise ValueError(f"LLM code changed parameters: {different}")
    return code


def Batch_Process_Queries(df, query_column='Query', log=LOG, cfg=CFG,
                          gold_column='Label-objective'):
    """Keep the direct LLM answer and directly extracted executable code."""
    results = []
    has_gold = gold_column in df.columns
    for pos, (idx, row) in enumerate(
            tqdm(list(df.iterrows()), desc="Processing Queries")):
        query = row[query_column]
        instance_id = int(idx) if isinstance(idx, int) else pos
        gold = row[gold_column] if has_gold else None
        category, raw_model_output = None, None
        with log.instance(instance_id=instance_id, query=query,
                          gold_objective=gold) as rec:
            try:
                NP_SET_CURRENT_QUERY(query)
                category, raw_model_output = Process_Input(query)
                executable_code = _extract_gurobipy_code(raw_model_output, query)
                rec.set(
                    pred_type=category,
                    model_output=raw_model_output,
                    code_output=executable_code,
                    code_assembly="direct_LLM_code_extraction",
                )
            except Exception as exc:
                print(f"[{instance_id}] failed: {type(exc).__name__}: {exc}")
                rec.fail(exc)
        results.append({
            "Category": category,
            "Original_Query": query,
            "Output": raw_model_output,
        })
    return pd.DataFrame(results)


In [ ]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import re
import numpy as np

def extract_objective(log_text):
   
    if "Optimal objective" in log_text:
        pattern = r"Optimal objective value: ([-+]?\d*\.?\d+)"
        match = re.search(pattern, log_text)
        if match:
            return float(match.group(1))

    elif "Best objective" in log_text:
        pattern = r"Best objective\s+([-+]?\d*\.?\d+e[-+]?\d+)"
        match = re.search(pattern, log_text, re.IGNORECASE)
        if match:
            return float(match.group(1))
    return None  



def gain_obj(df_out):
    output_code = df_out['Code'].tolist()
    obj = []
    i = 1

    for code in output_code:
        print(i)
        namespace = {'gp': gp, 'GRB': GRB}
        
  
        log_output = []
        

        class LogCapture:
            def write(self, message):
                log_output.append(message)

            def flush(self):  
                pass
            
        import sys
        log_capture = LogCapture()
        original_stdout = sys.stdout  
        sys.stdout = log_capture  

        try:
            exec(code, namespace)
        except Exception as e:
            log_output.append(f"Error: {e}\n")
            print(f"Error executing code block {i}: {e}")
        finally:
            sys.stdout = original_stdout 

    
        log_text = ''.join(log_output)

    
        optimal_value = extract_objective(log_text)
        if optimal_value is not None:
            obj.append(optimal_value)
            print(f"Optimal Value: {optimal_value}")
        else:

            obj.append("No optimal value found.")
            print("No optimal value found.")
        i += 1
    return obj


### Run CA

In [ ]:
# NP-only full-run cell. run_np_lean.py stops before this cell and selects rows safely.
df = pd.read_csv('Test_Dataset/Air_NRM/large_scale/query_NP_Flow.csv')
result_df = Batch_Process_Queries(df)
result_df.to_csv(LOG.dir / "answer_NP_Flow_largescale_GPT4.1.csv", index=False)

code_df = pd.DataFrame(
    [_extract_gurobipy_code(text, query)
     for text, query in zip(result_df['Output'], result_df['Original_Query'])],
    columns=['Code'],
)
code_df.to_csv(LOG.dir / "code_NP_Flow_largescale_GPT4.1.csv", index=False)
combined_df = pd.concat([result_df, code_df], axis=1)
combined_df.to_csv(LOG.dir / "final_NP_Flow_largescale_GPT4.1.csv", index=False)


### Run NP Flow

In [ ]:
# NP execution is defined in cell 18; no CA or small-scale fallback is present.


In [ ]:
# NP execution is defined in cell 18; no CA or small-scale fallback is present.


In [ ]:
# NP execution is defined in cell 18; no CA or small-scale fallback is present.


In [ ]:
# ---------------------------------------------------------------
# Finish the run: writes runs/<run_id>/summary.json and prints the
# totals. Safe to run after every slice.
# ---------------------------------------------------------------
agg = LOG.close()

import json
rows = [json.loads(l) for l in open(LOG.dir_path / 'instances.jsonl')]
est = sum(r['summary'].get('n_estimated_token_calls', 0) for r in rows)
print(f"\ntoken source: {'provider (good)' if est == 0 else f'{est} calls ESTIMATED -- report as estimates'}")
print('run dir:', LOG.dir_path)
